# The Day the AI Went Rogue  -  Day 1 Exercises

**Building Safe AI Systems**

This notebook holds the hands-on exercise for every segment of Day 1. Work top to bottom  - 
later segments reuse functions you build in earlier ones, and the API endpoint you build in
Segment 2 gets progressively upgraded as new tools become available (Segments 3, 4, and 6), so
run cells in order.

Cells marked **`# TODO`** are yours to fill in. Everything else is provided starter code, exactly as
it appears in the curriculum.


---
## Segment 1: Setup & Crisis
**Core concept:** Development Environment, Git Workflow

**Background:** Your engagement starts the same way any consulting engagement does  -  paperwork,
tooling, and access. Before you can fix anything, you need a working environment and a way to
track every change you make.

**Technical exercise:** this part happens in your terminal, not in this notebook. Run the commands
below in a terminal from your project folder.


In [ ]:
# This is a shell exercise, not Python  -  the cell below is for reference only.
# Copy these commands into your terminal:

setup_commands = """
git clone <starter-repo-url>
cd doctronic-rebuild
git checkout -b <your-name>-day1
echo "# Engagement started $(date)" >> LOG.md
git add LOG.md
git commit -m "Day 1: joined the rebuild engagement"
git push origin <your-name>-day1
"""
print(setup_commands)


**Prof checkpoint:** by end of segment, every student has (1) cloned the repo, (2) committed and
pushed a change.


---
## Segment 2: APIs with FastAPI and REST
**Core concept:** REST API Design, HTTP Endpoints, The Front Door Problem

**Background:** Doctronic's fabricated bulletin didn't arrive as a clean Python dictionary you
controlled  -  it arrived over the internet, from an outside party, through whatever interface was
listening. Before you write a single validation rule or business rule, you need a front door.
So that's what you're building first  -  deliberately, before anything else exists to protect it.

Right now, this endpoint accepts whatever shape of data shows up, no questions asked. That should
feel uncomfortable  -  it's close to what Doctronic effectively shipped. You'll close this gap
step by step, starting in the very next segment.

**Starter code:** the naive endpoint, plus an in-notebook `TestClient` so you can send requests
without needing a separate terminal running `uvicorn`.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()


@app.post("/renewal-request")
def submit_renewal(data: dict):
    # No shape enforced yet -- FastAPI will accept whatever keys and types show up here.
    # This is close to what Doctronic effectively had: an endpoint that trusts input by default.
    return {
        "status": "received",
        "patient_id": data.get("patient_id"),
        "medication": data.get("medication"),
        "dosage_mg": data.get("dosage_mg"),
        "source": data.get("source"),
    }


client = TestClient(app)

# Equivalent to:
# curl -X POST http://localhost:8000/renewal-request \
#   -H "Content-Type: application/json" \
#   -d '{"patient_id":"P001","age":"not-a-number","medication":"OxyContin","dosage_mg":30,"source":"Utah_Regulatory_Update_URGENT"}'

response = client.post("/renewal-request", json={
    "patient_id": "P001", "age": "not-a-number", "medication": "OxyContin",
    "dosage_mg": 30, "source": "Utah_Regulatory_Update_URGENT",
})
print(response.status_code, response.json())
# -> 200 OK. Nothing stopped a fabricated source, a malformed age, or a suspicious dosage.


**Student task:** send a payload with a fabricated `"source"` and a payload with a
wrong-typed field (e.g., age as text instead of a number). Confirm both are accepted without
complaint. Write one sentence: what's the actual production risk of shipping this version of the
endpoint?


In [ ]:
# TODO: send both payloads and print the status code + response for each

fabricated_source_payload = {
    "patient_id": "P002", "age": 45, "medication": "OxyContin",
    "dosage_mg": 10, "source": "Utah_Regulatory_Update_URGENT",
}

wrong_type_payload = {
    "patient_id": "P003", "age": "forty-five", "medication": "OxyContin",
    "dosage_mg": 10, "source": "FDA",
}

for payload in [fabricated_source_payload, wrong_type_payload]:
    r = client.post("/renewal-request", json=payload)
    print(payload["patient_id"], "->", r.status_code, r.json())


**Your one-sentence answer** (edit this cell): what's the production risk of shipping this
version of the endpoint? *TODO*


---
## Segment 3: Variables, Data Types & Validation
**Core concept:** Type Safety, Pydantic Schema Validation

**Background:** Mindgard's researchers didn't need to break through a firewall  -  they just handed
the chatbot a fabricated bulletin, and the system took it at face value. A type hint wouldn't have
caught this. You need something that actively enforces the shape of data, not just documents it  - 
exactly what your Segment 2 endpoint is currently missing.

This is also where Maria's specific failure mode gets caught. A fabricated *source* is one problem
 -  but even a **trusted** source can still request a dangerous dose. The model below checks both:
is this data from somewhere we trust, and does the dose itself make sense given what this patient
was prescribed before?

**Starter code:** the `PatientData` model with four validators wired in  -  three field-level checks
plus one model-level check that compares the new dose against the patient's previous dose.


In [ ]:
from pydantic import BaseModel, field_validator, model_validator, ValidationError

TRUSTED_SOURCES = {"FDA", "CDC", "State_Medical_Board"}

# Known safe dosage ranges per medication (mg)  -  in a real system this would come from
# a clinical reference database, not a hardcoded dict.
SAFE_DOSAGE_RANGES = {
    "oxycontin": (5, 30),   # typical range for chronic pain management
    "ibuprofen": (200, 800),
    "metformin": (500, 2000),
}


class PatientData(BaseModel):
    patient_id: str
    age: int
    medication: str
    dosage_mg: float
    previous_dosage_mg: float | None = None  # what they were prescribed last time, if known
    source: str

    @field_validator("age")
    @classmethod
    def age_must_be_reasonable(cls, v):
        if not (0 < v < 120):
            raise ValueError("age out of plausible range")
        return v

    @field_validator("source")
    @classmethod
    def source_must_be_trusted(cls, v):
        if v not in TRUSTED_SOURCES:
            raise ValueError(f"'{v}' is not a recognized trusted source")
        return v

    @field_validator("dosage_mg")
    @classmethod
    def dosage_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("dosage must be positive")
        return v

    # --- catches Maria's exact failure mode: a trusted source is not the same as a safe dose ---
    @model_validator(mode="after")
    def dosage_within_clinical_range_and_not_a_dangerous_jump(self):
        med = self.medication.lower()

        if med in SAFE_DOSAGE_RANGES:
            low, high = SAFE_DOSAGE_RANGES[med]
            if not (low <= self.dosage_mg <= high):
                raise ValueError(
                    f"{self.dosage_mg}mg of {self.medication} is outside "
                    f"the clinically accepted range ({low}-{high}mg)"
                )

        if self.previous_dosage_mg is not None and self.previous_dosage_mg > 0:
            multiplier = self.dosage_mg / self.previous_dosage_mg
            if multiplier >= 2:
                raise ValueError(
                    f"Dosage increased {multiplier:.1f}x from previous prescription "
                    f"({self.previous_dosage_mg}mg -> {self.dosage_mg}mg) "
                    f" -  flagged for pharmacist review"
                )
        return self


# Example 1: fabricated bulletin scenario (untrusted source)  -  structurally close to what
# actually happened to Doctronic.
bad_input_untrusted_source = {
    "patient_id": "P001",
    "age": 45,
    "medication": "OxyContin",
    "dosage_mg": 30,
    "previous_dosage_mg": 10,
    "source": "Utah_Regulatory_Update_URGENT",  # not in TRUSTED_SOURCES
}

try:
    PatientData(**bad_input_untrusted_source)
except ValidationError as e:
    print("Rejected (untrusted source):", e)

# Example 2: Maria's scenario  -  trusted source, but the dose itself is the problem.
# Notice the source check PASSES here. This is the case that would have slipped through
# Example 1's defense alone.
bad_input_dangerous_jump = {
    "patient_id": "P002",
    "age": 52,
    "medication": "OxyContin",
    "dosage_mg": 30,
    "previous_dosage_mg": 10,        # 3x jump from last prescription
    "source": "State_Medical_Board",  # trusted! passes the source check
}

try:
    PatientData(**bad_input_dangerous_jump)
except ValidationError as e:
    print("Rejected (dangerous jump):", e)


**Student task:** given a set of intentionally malformed patient records, extend
`PatientData` with 1–2 additional validators of your own design. For each, write one sentence:
what real-world Doctronic failure would this validator have caught?

*Ideas to consider (not already covered above): medication name against an allowlist, dosage
unit sanity, patient_id format.*


In [ ]:
# TODO: add your own validator(s) to a subclass (or edit PatientData directly) below.
# Example scaffold  -  replace with your own field and logic.

class PatientDataExtended(PatientData):
    pass

    # @field_validator("medication")
    # @classmethod
    # def medication_must_be_known(cls, v):
    #     # TODO: your logic here
    #     return v


# TODO: test your new validator(s) against a record designed to trip them
malformed_records = [
    # TODO: add 2-3 dicts here, each violating one of your new rules
]

for record in malformed_records:
    try:
        PatientDataExtended(**record)
        print("Accepted:", record)
    except ValidationError as e:
        print("Rejected:", record.get("patient_id"), "->", e)


**Your one-sentence explanations** (edit this cell):

- Validator 1 catches: *TODO*
- Validator 2 catches: *TODO*

### Callback: closing the Segment 2 gap

Your endpoint from Segment 2 accepted anything. Now that `PatientData` exists, close that gap by
changing what the endpoint expects.


In [ ]:
# Callback: rebuild the endpoint so FastAPI validates the body against PatientData
# before submit_renewal ever runs.

app = FastAPI()


@app.post("/renewal-request")
def submit_renewal(data: PatientData):
    # FastAPI validates the request body against PatientData before this function ever runs.
    return {"status": "received", "patient_id": data.patient_id}


client = TestClient(app)

# The exact payload that sailed through in Segment 2 now gets rejected at the door.
response = client.post("/renewal-request", json={
    "patient_id": "P001", "age": 45, "medication": "OxyContin",
    "dosage_mg": 30, "source": "Utah_Regulatory_Update_URGENT",
})
print(response.status_code)
print(response.json())
# -> 422 Unprocessable Entity


---
## Segment 4: Logic and Conditions
**Core concept:** Conditional Logic, Decision Trees, Business Rules as Code

**Background:** Valid data isn't the same as safe data. Even a perfectly well-formed request could
still be asking you to triple an opioid dose. Build gates that hold even if everything upstream
somehow gets fooled again.

**Starter code:** the base decision function.


In [ ]:
MAX_SAFE_DOSAGE = {
    "OxyContin": 20.0,  # mg, standard adult ceiling for renewal-without-review
}


def evaluate_renewal(data: PatientData) -> dict:
    if data.medication not in MAX_SAFE_DOSAGE:
        return {"approved": False, "reason": "medication not in approved renewal list"}

    if data.dosage_mg > MAX_SAFE_DOSAGE[data.medication]:
        return {"approved": False, "reason": "dosage exceeds hard safety ceiling  -  routed to physician"}

    if data.medication == "OxyContin":
        # controlled substances always get human review, no exceptions
        return {"approved": False, "reason": "controlled substance  -  requires physician sign-off"}

    return {"approved": True, "reason": "within safe parameters"}


**Student task:** given the three Mindgard scenarios below, confirm `evaluate_renewal` (or your
own added gate) stops each one  -  and write, in the markdown cell after, why that gate should hold
even if it arrived through a "trusted" source.

1. Tripled OxyContin dose
2. Methamphetamine reclassified as "unrestricted therapeutic"
3. Suspended-vaccine claim (a fabricated safety bulletin used to justify skipping a check)


In [ ]:
# TODO: build each scenario as a dict, run it through PatientData validation + evaluate_renewal,
# and print the outcome. Note: scenario 2 will likely need a new gate  -  add "Methamphetamine" to
# MAX_SAFE_DOSAGE or add explicit logic to evaluate_renewal to reject it outright.
#
# Note: these scenarios deliberately omit previous_dosage_mg. Segment 3's model_validator only
# catches a dangerous jump if it knows the patient's prior dose  -  Segment 4's job is to prove the
# hard ceiling holds even with NO prior-dose context at all.

scenario_tripled_dose = {
    "patient_id": "P100",
    "age": 45,
    "medication": "OxyContin",
    "dosage_mg": 30,  # tripled from the 10mg baseline
    "source": "FDA",
}

# scenario_meth_reclassified = {
#     "patient_id": "P101",
#     ...
#     "medication": "Methamphetamine",
#     ...
# }

# scenario_suspended_vaccine = {
#     ...
# }

scenarios = [scenario_tripled_dose]  # TODO: add the other two

for s in scenarios:
    try:
        patient = PatientData(**s)
        print(s["patient_id"], "->", evaluate_renewal(patient))
    except ValidationError as e:
        print(s["patient_id"], "-> rejected at validation:", e)


**Why each gate should hold regardless of source** (edit this cell):

1. Tripled dose: *TODO*
2. Reclassified controlled substance: *TODO*
3. Suspended-vaccine claim: *TODO*

### Callback: wiring the gate into the endpoint

Update your endpoint so a valid, well-formed request still gets evaluated against your safety
rules, not just accepted.


In [ ]:
# Callback: extend the endpoint to call evaluate_renewal

app = FastAPI()


@app.post("/renewal-request")
def submit_renewal(data: PatientData):
    decision = evaluate_renewal(data)
    return {"status": "received", "patient_id": data.patient_id, **decision}


client = TestClient(app)

# A well-formed but dangerous request now gets rejected for a different reason than a malformed one.
response = client.post("/renewal-request", json={
    "patient_id": "P100", "age": 45, "medication": "OxyContin",
    "dosage_mg": 30, "source": "FDA",  # legitimate source, dangerous dose
})
print(response.status_code, response.json())
# A 422 means "I don't understand this request."
# A safety rejection (200, approved=False) means "I understand it perfectly, and the answer is no."


---
## Segment 5: Loops and Automation
**Core concept:** Iteration, Batch Processing, Scaling Automation

**Background:** The clinic has 500 patients on chronic-condition renewals. Automation is genuinely
necessary  -  but one bug applied 5,000 times isn't a bug anymore, it's an incident.

**Starter code:** the batch processor.


In [ ]:
def process_batch(raw_records: list[dict]) -> list[dict]:
    results = []
    for record in raw_records:
        try:
            patient = PatientData(**record)
            decision = evaluate_renewal(patient)
            results.append({"patient_id": record.get("patient_id"), **decision, "status": "processed"})
        except ValidationError as e:
            results.append({"patient_id": record.get("patient_id"), "status": "rejected_at_validation", "reason": str(e)})
    return results


**Student task:** run the batch processor against a provided list of 20 mixed records (some
valid, some malformed, one modeled on the tripled-dosage scenario). Confirm: does one bad record
stop the rest? Produce a short summary of how many were approved, rejected, and routed to
physician review.


In [ ]:
import random

# Provided: 20 mixed records  -  valid, malformed, and one tripled-dosage scenario.
# TODO: feel free to shuffle / extend this list, but keep at least these characteristics.

random.seed(7)
mixed_records = []

# valid, low-dose, safe medication example
for i in range(12):
    mixed_records.append({
        "patient_id": f"P2{i:02d}",
        "age": random.randint(20, 80),
        "medication": "OxyContin",
        "dosage_mg": round(random.uniform(5, 20), 1),
        "source": "FDA",
    })

# malformed: bad source (the fabricated-bulletin pattern)
for i in range(4):
    mixed_records.append({
        "patient_id": f"P3{i:02d}",
        "age": 50,
        "medication": "OxyContin",
        "dosage_mg": 10.0,
        "source": "Utah_Regulatory_Update_URGENT",
    })

# malformed: missing field
mixed_records.append({"patient_id": "P400", "age": 40, "medication": "OxyContin", "source": "FDA"})

# malformed: wrong type
mixed_records.append({
    "patient_id": "P401", "age": "forty-five", "medication": "OxyContin",
    "dosage_mg": 10.0, "source": "FDA",
})

# the tripled-dosage scenario
mixed_records.append({
    "patient_id": "P999", "age": 45, "medication": "OxyContin",
    "dosage_mg": 30.0, "source": "FDA",
})

# a couple more malformed for good measure
mixed_records.append({
    "patient_id": "P402", "age": -5, "medication": "OxyContin",
    "dosage_mg": 10.0, "source": "FDA",
})

print(f"Total records: {len(mixed_records)}")


In [ ]:
results = process_batch(mixed_records)

# TODO: confirm one bad record doesn't halt the batch
assert len(results) == len(mixed_records), "Batch stopped early  -  one bad record broke the loop!"
print("✓ All records processed independently  -  one bad record did not stop the batch.")

# TODO: summarize approved / rejected / routed-to-physician counts
approved = sum(1 for r in results if r.get("approved") is True)
rejected_validation = sum(1 for r in results if r["status"] == "rejected_at_validation")
routed_to_physician = sum(1 for r in results if r.get("approved") is False)

print(f"Approved:                {approved}")
print(f"Rejected at validation:  {rejected_validation}")
print(f"Routed to physician:     {routed_to_physician}")


---
## Segment 6: Functions and Separation of Concerns
**Core concept:** Functions, Single Responsibility, Testable Units

**Background:** Your endpoint now handles real traffic, and your batch processor handles real
volume  -  which means the same validation-and-decision logic now lives in two places, written
twice. That's a live liability: a bug fixed in one copy can stay broken in the other, and there's
no way to test either piece on its own. Break the pipeline into small, single-purpose functions
that both the endpoint and the batch processor can share.

**Starter code:** three functions  -  `validate_request`, `decide_renewal`, `handle_request`.


In [ ]:
def validate_request(raw: dict) -> PatientData:
    """Single responsibility: turn raw input into a trusted, typed object."""
    return PatientData(**raw)  # raises ValidationError on bad shape


def decide_renewal(patient: PatientData) -> dict:
    """Single responsibility: apply business rules to already-valid data."""
    return evaluate_renewal(patient)


def handle_request(raw: dict) -> dict:
    """Composes the pieces above. If validation fails, decision logic never runs."""
    try:
        patient = validate_request(raw)
    except ValidationError as e:
        return {"status": "rejected_at_validation", "reason": str(e)}
    decision = decide_renewal(patient)
    return {"status": "processed", **decision}


**Student task:** write one unit test that feeds `validate_request` a bad record directly and
confirms `decide_renewal` is never called. This is the "isolation" claim  -  prove it, don't just
assert it.

*Hint: patch/monkeypatch `decide_renewal` with a call counter, then run `handle_request` on a
malformed record and assert the counter stayed at zero.*


In [ ]:
# TODO: implement the isolation test.
call_count = {"decide_renewal": 0}

def counting_decide_renewal(patient):
    call_count["decide_renewal"] += 1
    return decide_renewal(patient)


def test_bad_record_never_reaches_decision_logic():
    bad_record = {
        "patient_id": "P500",
        "age": 45,
        "medication": "OxyContin",
        "dosage_mg": 10,
        "source": "Utah_Regulatory_Update_URGENT",  # fails validation
    }

    try:
        validate_request(bad_record)
        raised = False
    except ValidationError:
        raised = True

    assert raised, "Expected validate_request to raise ValidationError on a bad record"

    # TODO: now confirm decide_renewal (or your counting wrapper) is never invoked
    # for this same bad record when run through the full handle_request path.
    assert call_count["decide_renewal"] == 0, "decide_renewal should never have been called"
    print("✓ Isolation confirmed: decide_renewal was never called for a bad record.")


test_bad_record_never_reaches_decision_logic()


### Callback: one set of functions, two callers

Point the endpoint at the same `handle_request` function your batch processor and your tests use,
instead of relying on FastAPI's automatic type-checking to do the work implicitly.


In [ ]:
# Callback: the endpoint now delegates entirely to handle_request

app = FastAPI()


@app.post("/renewal-request")
def submit_renewal(raw: dict):
    return handle_request(raw)


client = TestClient(app)

# One set of rules, enforced everywhere data enters the system -- not just at the HTTP layer.
response = client.post("/renewal-request", json={
    "patient_id": "P999", "age": 45, "medication": "OxyContin",
    "dosage_mg": 30, "source": "Utah_Regulatory_Update_URGENT",
})
print(response.status_code, response.json())


**Presentation checkpoint:** groups present what they've built so far  -  what AI tools/prompts
they used, what problems they hit.


---
## Segment 7: Databases and Audit Trails
**Core concept:** Relational Database Design, Schema Design, SQL Queries, Audit Logging

**Background:** If your system is ever asked "prove what happened, and prove you tried to prevent
it," an audit trail is your only real answer.

**Starter code:** `init_db` and `log_decision`.


In [ ]:
import sqlite3
from datetime import datetime

def init_db():
    conn = sqlite3.connect(":memory:")  # in-notebook: in-memory DB instead of audit.db on disk
    conn.execute("""
        CREATE TABLE IF NOT EXISTS decisions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id TEXT,
            status TEXT,
            reason TEXT,
            timestamp TEXT
        )
    """)
    conn.commit()
    return conn


def log_decision(conn, patient_id: str, status: str, reason: str):
    conn.execute(
        "INSERT INTO decisions (patient_id, status, reason, timestamp) VALUES (?, ?, ?, ?)",
        (patient_id, status, reason, datetime.utcnow().isoformat())
    )
    conn.commit()


**Student task:** wire `log_decision` into `handle_request` from Segment 6 so every outcome  - 
approved, rejected, routed to physician  -  is logged automatically. Then run a query a regulator
might ask for: "show me every controlled-substance request from today and why each was routed to
physician review."


In [ ]:
# TODO: this rewires handle_request to take a `conn` and log every outcome.

def handle_request(raw: dict, conn) -> dict:
    try:
        patient = validate_request(raw)
    except ValidationError as e:
        result = {"status": "rejected_at_validation", "reason": str(e)}
        log_decision(conn, raw.get("patient_id", "unknown"), result["status"], result["reason"])
        return result

    decision = decide_renewal(patient)
    result = {"status": "processed", **decision}
    log_decision(conn, patient.patient_id, result["status"], result.get("reason", ""))
    return result


# Run the whole mixed_records batch (Segment 5) through the logged pipeline
conn = init_db()
for record in mixed_records:
    handle_request(record, conn)

import pandas as pd
audit_df = pd.read_sql_query("SELECT * FROM decisions", conn)
audit_df


In [ ]:
# TODO: the regulator's query  -  every controlled-substance request routed to physician review
controlled_substance_flags = pd.read_sql_query("""
    SELECT patient_id, reason, timestamp
    FROM decisions
    WHERE reason LIKE '%controlled substance%'
    ORDER BY timestamp DESC
""", conn)

controlled_substance_flags


---
## Segment 8: Testing and Governance
**Core concept:** Unit Testing, Adversarial Testing, Catching Vulnerabilities

**Background:** The real test isn't "does it work on clean data," it's "does it survive someone
actively trying to fool it, the way Mindgard did."

**Starter code:** the first adversarial test, modeled on the fabricated bulletin.


In [ ]:
def test_fabricated_bulletin_rejected():
    conn = init_db()

    fabricated_request = {
        "patient_id": "P999",
        "age": 45,
        "medication": "OxyContin",
        "dosage_mg": 30,  # tripled dose, mirrors the real Mindgard exploit
        "source": "Utah_Regulatory_Update_URGENT",  # fabricated bulletin
    }

    result = handle_request(fabricated_request, conn)

    assert result["status"] == "rejected_at_validation"

    # confirm the rejection was actually logged, not just returned
    row = conn.execute(
        "SELECT * FROM decisions WHERE patient_id = 'P999'"
    ).fetchone()
    assert row is not None

    print("✓ test_fabricated_bulletin_rejected passed")


test_fabricated_bulletin_rejected()


**Student task:** write two more adversarial tests modeled on the other real Mindgard findings  - 
methamphetamine reclassified as "unrestricted therapeutic," and the suspended-vaccine claim. For
each, confirm both that the system rejects it and that the rejection is logged.


In [ ]:
# TODO: adversarial test #2  -  meth reclassified as "unrestricted therapeutic"

def test_reclassified_controlled_substance_rejected():
    conn = init_db()

    fabricated_request = {
        "patient_id": "P998",
        "age": 45,
        "medication": "Methamphetamine",
        "dosage_mg": 10,
        "source": "Utah_Regulatory_Update_URGENT",  # claims meth is "unrestricted therapeutic"
    }

    result = handle_request(fabricated_request, conn)

    # TODO: assert the request was rejected (at validation, since "Methamphetamine" + this source
    # both fail existing checks)  -  adjust if your Segment 4 gates handle this differently.
    assert result["status"] in ("rejected_at_validation",) or result.get("approved") is False

    row = conn.execute("SELECT * FROM decisions WHERE patient_id = 'P998'").fetchone()
    assert row is not None

    print("✓ test_reclassified_controlled_substance_rejected passed")


test_reclassified_controlled_substance_rejected()


In [ ]:
# TODO: adversarial test #3  -  suspended-vaccine claim used to justify skipping a safety check

def test_suspended_vaccine_claim_rejected():
    conn = init_db()

    fabricated_request = {
        "patient_id": "P997",
        "age": 45,
        "medication": "OxyContin",
        "dosage_mg": 25,  # above the safety ceiling
        "source": "Utah_Regulatory_Update_URGENT",  # fabricated "vaccine suspended, skip review" bulletin
    }

    result = handle_request(fabricated_request, conn)

    assert result["status"] == "rejected_at_validation"

    row = conn.execute("SELECT * FROM decisions WHERE patient_id = 'P997'").fetchone()
    assert row is not None

    print("✓ test_suspended_vaccine_claim_rejected passed")


test_suspended_vaccine_claim_rejected()


**Presentation checkpoint:** groups present their final system  -  what AI tools/prompts they
used, what problems they hit, and a live run of their adversarial tests.


---
## Segment 9: Introduction to Data Visualization
**Core concept:** Grammar of Good Visualization, Misleading Charts, Storytelling Frameworks

**Background:** Doctronic reported a 99.2% match rate with clinicians across 500 urgent-care cases.
That statistic is true  -  and also easy to turn into a misleading chart.

**Technical exercise:** below are two versions of the same chart. One is built to make the 99.2%
figure look as reassuring as possible; one is built honestly.


In [ ]:
import matplotlib.pyplot as plt

match_rate = 99.2
n = 500

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Version 1: the misleading version  -  truncated axis, no sample size, no comparison
axes[0].bar(["Doctronic"], [match_rate], color="seagreen")
axes[0].set_ylim(98, 100)  # truncated axis exaggerates how close to perfect this looks
axes[0].set_title("Clinician Match Rate")  # states the topic, not the finding
axes[0].set_ylabel("%")

# Version 2: the honest version  -  full axis, sample size labeled, comparison shown
axes[1].bar(["Doctronic\n(n=500)"], [match_rate], color="seagreen")
axes[1].set_ylim(0, 100)  # full, honest scale
axes[1].set_title("Doctronic Matched Clinicians 99.2% of the Time\n(n=500, single urgent-care sample)")
axes[1].set_ylabel("%")

plt.tight_layout()
plt.show()


**Student task:** identify every specific choice that makes Version 1 misleading, and write the
corrected title/labels for Version 2 (edit this cell).

Misleading choices in Version 1:
- TODO
- TODO
- TODO

Corrected title for Version 2: *TODO*
Corrected axis label(s): *TODO*


---
## Segment 10: Building Charts  -  Matplotlib, Seaborn, Plotly
**Core concept:** Chart Selection, Static vs. Interactive Visualization

**Background:** Turn your `audit.db`  -  the same database you built the audit trail into  -  into the
monthly report the state contract requires.

**Starter code:** pull data straight from the in-notebook audit DB (built in Segment 7) and build
three report-ready charts.


In [ ]:
import seaborn as sns
import plotly.express as px

df = pd.read_sql_query("SELECT * FROM decisions", conn)
df["date"] = df["timestamp"].str[:10]

# 1. Matplotlib  -  approvals vs rejections over time (trend -> line chart)
daily = df.groupby(["date", "status"]).size().unstack(fill_value=0)
daily.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("Daily Renewal Decisions  -  Approved vs Rejected")
plt.ylabel("Number of Requests")
plt.show()

# 2. Seaborn  -  distribution of rejection reasons (categorical comparison -> bar)
plt.figure(figsize=(7, 4))
sns.countplot(data=df[df["status"] != "processed"], y="reason")
plt.title("Why Requests Were Rejected or Flagged This Month")
plt.tight_layout()
plt.show()

# 3. Plotly  -  interactive drill-down for the regulator to explore themselves
fig = px.histogram(df, x="date", color="status",
                    title="Renewal Volume by Outcome (interactive)")
fig.show()


**Student task:** for each of the three charts, write one sentence justifying the chart type
chosen  -  why a line here, why a bar there, why interactivity matters for the third  -  tying the
choice back to the specific question a regulator would be asking.

- Matplotlib (line): *TODO*
- Seaborn (bar): *TODO*
- Plotly (interactive histogram): *TODO*


---
## Segment 11: Streamlit  -  The Living Dashboard
**Core concept:** Rapid App Building, Always-Current Reporting

**Background:** A monthly PDF report is already out of date the day after it's sent. The regulator
needs something they can check on their own schedule.

**Note:** Streamlit apps don't run inline in a notebook  -  they run as a standalone script via
`streamlit run app.py`. The cell below writes the starter app to a file so you can launch it from
a terminal. Play with the filters first, then make the refinements in the student task.


In [ ]:
streamlit_app_code = '''
import streamlit as st
import sqlite3
import pandas as pd

st.title("Prescription Renewal Oversight Dashboard")

conn = sqlite3.connect("audit.db")
df = pd.read_sql_query("SELECT * FROM decisions", conn)

status_filter = st.multiselect("Filter by status", df["status"].unique(), default=df["status"].unique())
filtered = df[df["status"].isin(status_filter)]

st.metric("Total requests logged", len(df))
st.metric("Flagged for physician review", len(df[df["status"] != "processed"]))

st.bar_chart(filtered["status"].value_counts())
st.dataframe(filtered.sort_values("timestamp", ascending=False))
'''

with open("dashboard_app.py", "w") as f:
    f.write(streamlit_app_code)

print("Wrote dashboard_app.py  -  run it with: streamlit run dashboard_app.py")
print("(Make sure audit.db exists on disk in the same folder  -  export your in-memory conn if needed.)")


**Student task  -  play & refine:** open `dashboard_app.py`, run it with `streamlit run
dashboard_app.py`, and explore the filters. Then add the following directly in that file
(not in this notebook):

1. A date-range filter so a regulator can inspect a specific week.
2. A visible callout when flagged requests exceed a threshold you define (e.g., "5+
   controlled-substance flags this week  -  review recommended").
3. **Stretch goal:** surface the actual reason text next to each flagged row, so the regulator
   never has to ask "why."

TODO scaffold for the date-range filter (paste into `dashboard_app.py`):


In [ ]:
todo_snippet = '''
# TODO: add near the top of dashboard_app.py, after loading df
df["timestamp"] = pd.to_datetime(df["timestamp"])
min_date, max_date = df["timestamp"].min().date(), df["timestamp"].max().date()
date_range = st.date_input("Date range", value=(min_date, max_date))

# TODO: filter `filtered` further using date_range before displaying metrics/charts
'''
print(todo_snippet)


---
## Closing: Ship It

By the end of this notebook, your system rejects malformed data at the door, refuses dangerous
decisions regardless of how convincing the source looks, logs every decision it makes, has been
tested against the exact attack that got the original pilot suspended, and reports its own
performance honestly.

Commit this notebook to your repo alongside your `dashboard_app.py`  -  this is part of your
portfolio-ready deliverable.
